# 09 · 작은 CNN 대조

**질문:** 같은 산모·구간에서 Cat28과 파형 CNN의 성능은 어떻게 다른가? CNN의 폭·정규화·학습 상태에 따라 그 차이가 달라지는가?

02의 `cnn_signals.npy`를 `features.csv`의 `signal_index`로 골라 사용한다. 이 파형은 **FHR 결측을 보간하고 TOCO는 원본을 유지한** 입력이다. 01의 산모 분할과 04의 Cat28 기준 예측을 그대로 사용한다.

저장소의 CTGNetMini 구조를 길이 150에 맞춰 처음부터 학습한다. 사전학습 가중치를 사용하는 실험은 아니다. 폭과 대표 seed는 validation AP로만 선택하고, 선택 뒤 test를 평가한다.

## 1. 실행 예산과 입력

기본값은 원본의 로컬 CPU 예산이다. 두 정규화 × 폭 3개 × seed 3개 = 18회이며, 최대 400 epoch·최소 40 epoch·patience 40을 사용한다. `FULL_CNN=True`는 폭 16·128을 추가하여 30회, 최대 1,000 epoch·patience 150으로 바꾼다. 예산은 결과를 보기 전에 정한다.

데이터·설정이 같은 재실행은 `data/results/cnn/`의 완료 작업과 중단 체크포인트를 재사용한다. 입력·예산이 달라지면 이전 폴더를 별도로 보관하고 새 실험으로 시작해야 한다. 이번 노트북은 이를 확인하는 `cnn/run.json`을 추가한다.

In [ ]:
from pathlib import Path
import hashlib
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import average_precision_score

RESULTS = Path("data/results")
CNN = RESULTS / "cnn"
FULL_CNN = False
SEEDS = (42, 43, 44)
MODES = ("channel_maxabs", "per_segment_z")
WIDTHS = (8, 16, 32, 64, 128) if FULL_CNN else (8, 32, 64)
MAX_EPOCHS = 1000 if FULL_CNN else 400
MIN_EPOCHS = 40
PATIENCE = 150 if FULL_CNN else 40
BATCH_SIZE = 256
THREADS = 2
N_BOOTSTRAP = 2000
BOOTSTRAP_SEED = 42
ID_TYPES = {key: str for key in ("row_id", "record_id", "mother_id", "site")}

### 출력에 남기는 오류 정보

오류는 고정된 분류와 건수로만 요약한다. `other_error`의 상세 원인은 내부 오류 파일에서 확인한다. 건수는 오류 내역 수이며, 같은 기록이 여러 번 포함될 수 있다.

In [ ]:
def issue_category(reason):
    # Match known phrases, but return ONLY fixed labels: never echo an exception.
    # Example: "FileNotFoundError: /private/DEMO.json" -> "missing_input".
    text = str(reason)
    groups = {
        "missing_input": ("FileNotFoundError", "No such file", "[Errno 2]"),
        "missing_dependency": ("No module named", "cannot import name", "ImportError"),
        "fhr_missing": ("FHR 결측률",),
        "invalid_wave": ("FHR과 TOCO", "측정 간격", "FHR은 0 이상", "신호 길이"),
        "invalid_bbox": ("Bbox",),
        "invalid_id": ("ID가", "산모 ID", "산모 목록", "행 정보 불일치", "구간 정보 불일치"),
        "invalid_labels": ("라벨이 올바르지", "라벨 수", "라벨·점수"),
        "no_usable_data": ("구간이 없습니다", "비었", "남은 산모가 없습니다"),
        "split_unavailable": ("분할", "양성·음성", "정상·비정상", "least populated class", "test_size"),
        "checkpoint_mismatch": ("체크포인트", "CNN 입력·설정"),
        "io_error": ("PermissionError", "Permission denied", "No space left", "OSError"),
        "invalid_value": ("ValueError", "TypeError", "유한", "could not convert", "ufunc"),
    }
    for category, phrases in groups.items():
        if any(phrase in text for phrase in phrases):
            return category
    return "other_error"

In [ ]:
def show_issue_counts(reasons):
    # Count issue entries, not unique people; one record may have several entries.
    categories = pd.Series([issue_category(reason) for reason in reasons], dtype="string")
    counts = categories.value_counts().rename_axis("category").to_frame("count")
    print("제외·오류 내역 수 =", len(categories))
    # Full messages, IDs, paths, and jobs stay in the internal CSV files.
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        display(counts)

In [ ]:
PREDICTION_COLUMNS = [
    "row_id", "record_id", "mother_id", "site", "target", "split",
    "record_all_normal", "signal_index", "score", "threshold",
]
METRIC_COLUMNS = ["metric", "point", "low", "high", "valid_bootstrap", "mothers", "segments"]
TRIAL_COLUMNS = [
    "mode", "width", "seed", "val_AP", "best_epoch", "epochs_run", "hit_cap",
    "patience_met", "stale_epochs", "parameters", "fit_seconds",
]
HISTORY_COLUMNS = ["epoch", "train_loss", "val_loss", "val_AP", "learning_rate"]
COST_COLUMNS = ["mode", "width", "seed", "microseconds_per_window", "parameters"]
issues = []
ready = False
setup_error = None
print("planned trials =", len(MODES) * len(WIDTHS) * len(SEEDS))
print("widths =", WIDTHS, "max epochs =", MAX_EPOCHS, "patience =", PATIENCE)

In [ ]:
torch_available = False
try:
    import torch
    from torch import nn
    torch.set_num_threads(THREADS)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch_available = True
    print("device =", device, "torch =", torch.__version__)
except (ImportError, OSError, RuntimeError) as error:
    setup_error = str(error)
    issues.append(dict(stage="setup", job="torch", reason=setup_error))
    print('PyTorch를 이용할 수 없습니다:', '[details omitted from notebook output]')

In [ ]:
def split_holdout(frame):
    parts = []
    for split in ("train", "val", "test"):
        # This reads saved assignments; it does not draw a fresh split per arm.
        part = frame.loc[frame["split"].eq(split)].copy()
        # Example: [0, 0, 0] alone cannot support the planned binary comparison.
        if set(part["target"].unique()) != {0, 1}:
            raise ValueError("train/val/test 각각 정상·비정상 라벨이 필요합니다")
        parts.append(part)
    return parts

In [ ]:
if torch_available:
    try:
        protocol = json.loads((RESULTS / "protocol.json").read_text(encoding="utf-8"))
        if not protocol.get("split_ready"):
            raise ValueError("01의 산모 분할이 완료되지 않았습니다")
        frame = pd.read_csv(RESULTS / "features.csv", dtype=ID_TYPES)
        train, validation, test = split_holdout(frame)
        if frame["row_id"].duplicated().any():
            raise ValueError("구간 ID가 중복됩니다")
        signal_array = np.load(RESULTS / "cnn_signals.npy", mmap_mode="r")
        indices = frame["signal_index"].to_numpy()
        if not np.isfinite(indices).all() or not np.equal(indices, np.floor(indices)).all():
            raise ValueError("signal_index는 정수여야 합니다")
        if (indices < 0).any() or (indices >= len(signal_array)).any():
            raise ValueError("signal_index가 파형 배열 범위를 벗어납니다")
        # Example: feature rows with signal_index [0, 2] select waves 0 and 2, not 0 and 1.
        raw = signal_array[indices.astype(int)]
        if raw.shape != (len(frame), 2, 150) or not np.isfinite(raw).all():
            raise ValueError("CNN 입력은 구간별 유한한 (2, 150) 배열이어야 합니다")
        masks = {split: frame["split"].eq(split).to_numpy() for split in ("train", "val", "test")}
        labels = frame["target"].to_numpy(np.float32)
        ready = True
    except Exception as error:
        setup_error = str(error)
        issues.append(dict(stage="setup", job="input", reason=setup_error))
        print('CNN 입력을 준비할 수 없습니다:', '[details omitted from notebook output]')

In [ ]:
if ready:
    print("selected signal shape =", raw.shape)
    print("train / validation / test windows =", len(train), len(validation), len(test))

## 2. 정규화 두 가지를 구분한다

`channel_maxabs`는 훈련 파형 전체에서 채널별 최대 절댓값을 구해 나눈다. 두 채널은 서로 다른 분모를 쓰며, validation·test도 훈련 분모를 사용한다.

`per_segment_z`는 구간·채널마다 시간축 평균과 표준편차로 정규화한다. 각 구간의 평균 심박수와 절대 크기 정보가 사라진다. 분모는 최소 `1e-6`으로 제한한다.

In [ ]:
def normalize(waves, mode, scale):
    if mode == "channel_maxabs":
        # Example: FHR 150 / training FHR max 200 -> 0.75.
        # scale shape (2,) becomes (1, 2, 1) for windows x channels x time.
        return (waves / scale[None, :, None]).astype(np.float32)
    if mode != "per_segment_z":
        raise ValueError("알 수 없는 정규화: " + mode)
    # Example: a constant channel [140, 140, ...] becomes all zeros.
    mean = waves.mean(axis=2, keepdims=True)
    deviation = np.maximum(waves.std(axis=2, keepdims=True), 1e-6)
    return ((waves - mean) / deviation).astype(np.float32)

In [ ]:
example_waves = np.array([[[100., 150., 200.], [10., 10., 10.]]])
example_scale = np.array([200., 20.])
# maxabs -> FHR [0.5, 0.75, 1.0], TOCO [0.5, 0.5, 0.5].
# z -> FHR near [-1.225, 0, 1.225], TOCO [0, 0, 0].
print("maxabs example =", normalize(example_waves, "channel_maxabs", example_scale).tolist())
print("z example =", normalize(example_waves, "per_segment_z", example_scale).round(3).tolist())
if ready:
    # Reduce the window and time axes, leaving one training maximum per channel.
    scale = np.maximum(np.abs(raw[masks["train"]]).max(axis=(0, 2)), 1e-6)
    print("training channel scales [FHR, TOCO] =", scale.tolist())

## 3. CNN의 입력과 층을 따라간다

입력은 `(batch, 2, 150)`이다. 폭 `width`는 중간 채널 수이며, 시간 길이는 `150 → 136 → 122 → 61 → 47 → 23`으로 줄어든다. 마지막에는 구간당 logit 하나를 만들고 sigmoid로 확률로 바꾼다.

구조는 `02-반입할-파일/fg/ctgnet.py`를 길이 150에 맞춘 원본 09와 같다. 기존 파일의 기본 길이 157에 붙는 통계량 7개는 여기서 사용하지 않는다.

In [ ]:
if torch_available:
    class CTGNetMini(nn.Module):
        def __init__(self, width):
            super().__init__()
            # Kernel 15 spans 15 samples; at 2-second sampling this is a 30-second scale.
            self.conv1 = nn.Conv1d(2, width, 15)
            self.bn1 = nn.BatchNorm1d(width)
            # groups=width: each learned channel gets its own temporal filter.
            self.conv2 = nn.Conv1d(width, width, 15, groups=width)
            self.bn2 = nn.BatchNorm1d(width)
            self.conv3_dw = nn.Conv1d(width, width, 15, groups=width)
            self.conv3_pw = nn.Conv1d(width, width, 1)  # Mix channels at each time step.
            self.bn3 = nn.BatchNorm1d(width)
            self.act = nn.ELU()
            self.pool = nn.AvgPool1d(2)
            self.drop = nn.Dropout(0.25)
            self.fc = nn.Linear(width * 23, 1)

        def forward(self, waves):
            # (B, 2, 150) -> (B, width, 136).
            waves = self.bn1(self.conv1(waves))
            # Depthwise convolution: 136 -> 122; average pooling: 122 -> 61.
            waves = self.bn2(self.conv2(waves))
            waves = self.drop(self.pool(self.act(waves)))
            # Depthwise + pointwise: 61 -> 47; pooling: 47 -> 23.
            waves = self.bn3(self.conv3_pw(self.conv3_dw(waves)))
            waves = self.drop(self.pool(self.act(waves)))
            # (B, width, 23) -> (B, width*23) -> (B,) logits.
            return self.fc(torch.flatten(waves, 1)).squeeze(1)

In [ ]:
if torch_available:
    # Use a separate CPU example; it neither trains nor supplies trial weights.
    example_model = CTGNetMini(width=8).eval()
    with torch.no_grad():
        example_logits = example_model(torch.zeros(3, 2, 150))
    print("example input shape =", (3, 2, 150))
    print("example output shape =", tuple(example_logits.shape))
    # Parameter count = width^2 + 93*width + 1; width 8 -> 809.
    print("width 8 parameters =", sum(value.numel() for value in example_model.parameters()))

In [ ]:
def scores(model, waves):
    model.eval()  # Disable dropout and use stored batch-normalization statistics.
    chunks = []
    with torch.no_grad():
        for start in range(0, len(waves), 1024):
            batch = torch.from_numpy(waves[start:start + 1024]).to(device)
            # Example: logit 0 -> sigmoid 0.5, one abnormal-reading score per window.
            chunks.append(torch.sigmoid(model(batch)).cpu().numpy())
    return np.concatenate(chunks)

## 4. 체크포인트의 입력과 설정을 확인한다

같은 폴더의 모델을 이어 쓸 때 입력 파일과 학습 설정이 같아야 한다. 원본에는 없던 입력·설정 확인을 추가한다. 일치하지 않으면 학습을 미산출로 남기며 기존 체크포인트를 덮어쓰지 않는다.

`best.pt`는 최적 검증 AP의 가중치, `last.pt`는 가장 최근 epoch의 모델·옵티마이저·학습률·난수 상태다. `complete.json`이 있는 작업은 재학습 없이 읽는다.

In [ ]:
def file_digest(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        # Read large arrays in chunks rather than making another whole-array copy.
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

In [ ]:
def check_run_manifest():
    manifest = dict(
        version=1, features_sha256=file_digest(RESULTS / "features.csv"),
        signals_sha256=file_digest(RESULTS / "cnn_signals.npy"),
        widths=list(WIDTHS), seeds=list(SEEDS), modes=list(MODES),
        max_epochs=MAX_EPOCHS, min_epochs=MIN_EPOCHS, patience=PATIENCE,
        batch_size=BATCH_SIZE, threads=THREADS, learning_rate=3e-4, weight_decay=1e-4,
        device=str(device), torch_version=str(torch.__version__), numpy_version=np.__version__,
    )
    path = CNN / "run.json"
    if path.exists():
        if json.loads(path.read_text(encoding="utf-8")) != manifest:
            raise ValueError("CNN 입력·설정이 달라졌습니다. 기존 data/results/cnn 폴더를 별도 보관한 뒤 다시 실행하세요")
    else:
        if any(CNN.glob("*/last.pt")) or any(CNN.glob("*/complete.json")):
            raise ValueError("실행 정보가 없는 체크포인트가 있습니다. 기존 cnn 폴더를 별도 보관하세요")
        CNN.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return manifest

In [ ]:
if ready:
    try:
        run_manifest = check_run_manifest()
        print("checkpoint inputs and settings match")
    except (OSError, ValueError) as error:
        ready = False
        setup_error = str(error)
        issues.append(dict(stage="setup", job="checkpoint", reason=setup_error))
        print('체크포인트를 사용할 수 없습니다:', '[details omitted from notebook output]')

In [ ]:
def atomic_save(path, value):
    # Write a temporary file first; a stopped write must not replace last.pt halfway.
    temporary = path.with_suffix(".tmp")
    torch.save(value, temporary)
    temporary.replace(path)

## 5. 학습 한 epoch와 검증을 정의한다

양성 가중치는 훈련 구간에서 `sqrt(음성 수 / 양성 수)`로 계산한다. AdamW의 학습률은 `3e-4`, weight decay는 `1e-4`이고 cosine schedule을 사용한다.

훈련 loss는 미니배치의 구간 수로 가중 평균한다. validation loss·AP를 기록하지만 최적 epoch는 validation AP로 정한다. test는 아래 학습 함수에 들어가지 않는다.

In [ ]:
def training_objects(width, seed):
    torch.manual_seed(seed)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(seed)
    model = CTGNetMini(width).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
    positive = labels[masks["train"]].sum()
    negative = masks["train"].sum() - positive
    # Example: 900 negative and 100 positive windows -> positive weight sqrt(9)=3.
    weight = np.sqrt(negative / positive)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(weight, device=device))
    return model, optimizer, scheduler, criterion

In [ ]:
def train_one_epoch(model, optimizer, criterion, waves, seed, epoch):
    model.train()
    # Reproducible shuffle per epoch, independent of the checkpoint restart time.
    # Example: epoch 0 uses seed 42; epoch 1 uses 42+10007 for trial seed 42.
    rng = np.random.default_rng(seed + epoch * 10007)
    order = rng.permutation(np.flatnonzero(masks["train"]))
    total_loss = 0.0
    for start in range(0, len(order), BATCH_SIZE):
        indices = order[start:start + BATCH_SIZE]
        optimizer.zero_grad(set_to_none=True)
        batch = torch.from_numpy(waves[indices]).to(device)
        targets = torch.from_numpy(labels[indices]).to(device)
        loss = criterion(model(batch), targets)
        if not torch.isfinite(loss):
            raise ValueError("학습 손실이 유한하지 않습니다")
        loss.backward()
        optimizer.step()
        # The final shorter batch gets proportionally less weight in the epoch mean.
        total_loss += loss.item() * len(indices)
    return total_loss / len(order)

In [ ]:
def validation_loss(model, criterion, waves):
    model.eval()
    indices = np.flatnonzero(masks["val"])
    total_loss = 0.0
    with torch.no_grad():
        for start in range(0, len(indices), 1024):
            batch_indices = indices[start:start + 1024]
            batch = torch.from_numpy(waves[batch_indices]).to(device)
            targets = torch.from_numpy(labels[batch_indices]).to(device)
            loss = criterion(model(batch), targets)
            total_loss += loss.item() * len(batch_indices)
    return total_loss / len(indices)

## 6. 중단 시점에서 재개하고 최적 epoch를 남긴다

검증 AP가 기존 최고보다 `1e-8`을 초과하여 높아져야 개선으로 센다. 최소 40 epoch를 수행한 뒤, 연속 미개선 횟수가 patience에 도달하면 종료한다. 예를 들어 최고 epoch가 20이고 이후 40번 개선되지 않으면 60 epoch를 마친 뒤 멈춘다.

재개할 때 가중치뿐 아니라 옵티마이저·scheduler·PyTorch 난수 상태를 복원한다. batch 순서는 seed와 epoch에서 다시 계산한다.

In [ ]:
def restore_progress(folder, model, optimizer, scheduler):
    progress = dict(best=-1.0, best_epoch=0, stale=0, epoch=0, elapsed=0.0, history=[])
    checkpoint = folder / "last.pt"
    if not checkpoint.exists():
        return progress
    state = torch.load(checkpoint, map_location=device, weights_only=True)
    model.load_state_dict(state["model"])
    optimizer.load_state_dict(state["optimizer"])
    scheduler.load_state_dict(state["scheduler"])
    for key in progress:
        progress[key] = state[key]
    # Example: saved epoch=12 -> start the zero-based loop at 12 (the 13th epoch).
    torch.set_rng_state(state["rng"].cpu())
    if device.type == "cuda" and state["cuda_rng"]:
        torch.cuda.set_rng_state_all([value.cpu() for value in state["cuda_rng"]])
    return progress

In [ ]:
def finish_epoch(folder, model, optimizer, scheduler, progress, row, elapsed):
    progress["history"].append(row)
    if row["val_AP"] > progress["best"] + 1e-8:
        progress["best"] = row["val_AP"]
        progress["best_epoch"] = row["epoch"]
        progress["stale"] = 0
        atomic_save(folder / "best.pt", {key: value.detach().cpu() for key, value in model.state_dict().items()})
    else:
        progress["stale"] += 1
    scheduler.step()
    progress["epoch"] = row["epoch"]
    progress["elapsed"] = elapsed
    # Save the scheduler AFTER its step, ready for the next epoch's learning rate.
    checkpoint = dict(
        model=model.state_dict(), optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(),
        **progress, rng=torch.get_rng_state(),
        cuda_rng=torch.cuda.get_rng_state_all() if device.type == "cuda" else [],
    )
    atomic_save(folder / "last.pt", checkpoint)
    pd.DataFrame(progress["history"], columns=HISTORY_COLUMNS).to_csv(folder / "history.csv", index=False)

In [ ]:
def trial_summary(model, progress, mode, width, seed, elapsed):
    epochs_run = len(progress["history"])
    # hit_cap=True and patience_met=False means the budget ended before the plateau rule.
    # Patience stopping itself does not prove global optimization convergence.
    return dict(
        mode=mode, width=width, seed=seed, val_AP=progress["best"],
        best_epoch=progress["best_epoch"], epochs_run=epochs_run,
        hit_cap=epochs_run >= MAX_EPOCHS, patience_met=progress["stale"] >= PATIENCE,
        stale_epochs=progress["stale"], parameters=sum(value.numel() for value in model.parameters()),
        fit_seconds=elapsed,
    )

In [ ]:
def fit_trial(waves, mode, width, seed):
    folder = CNN / f"{mode}_w{width}_s{seed}"
    folder.mkdir(parents=True, exist_ok=True)
    completed = folder / "complete.json"
    if completed.exists():
        if not (folder / "best.pt").exists() or not (folder / "history.csv").exists():
            raise ValueError("완료 표시가 있지만 가중치 또는 학습 이력이 없습니다")
        return json.loads(completed.read_text(encoding="utf-8"))
    model, optimizer, scheduler, criterion = training_objects(width, seed)
    progress = restore_progress(folder, model, optimizer, scheduler)
    previous_elapsed = progress["elapsed"]
    started = time.perf_counter()
    for epoch in range(progress["epoch"], MAX_EPOCHS):
        if epoch >= MIN_EPOCHS and progress["stale"] >= PATIENCE:
            break
        learning_rate = optimizer.param_groups[0]["lr"]
        train_loss = train_one_epoch(model, optimizer, criterion, waves, seed, epoch)
        val_loss = validation_loss(model, criterion, waves)
        val_ap = float(average_precision_score(labels[masks["val"]], scores(model, waves[masks["val"]])))
        row = dict(epoch=epoch + 1, train_loss=train_loss, val_loss=val_loss,
                   val_AP=val_ap, learning_rate=learning_rate)
        elapsed = previous_elapsed + time.perf_counter() - started
        finish_epoch(folder, model, optimizer, scheduler, progress, row, elapsed)
        if (epoch + 1) % 20 == 0:
            print(mode, width, seed, "epoch", epoch + 1, "best val AP", round(progress["best"], 4), flush=True)
    elapsed = previous_elapsed + time.perf_counter() - started
    result = trial_summary(model, progress, mode, width, seed, elapsed)
    completed.write_text(json.dumps(result, indent=2), encoding="utf-8")
    print("CNN complete:", mode, width, seed, "| best epoch", result["best_epoch"], flush=True)
    return result

In [ ]:
trials = []
if ready:
    for mode in MODES:
        waves = normalize(raw, mode, scale)
        for width in WIDTHS:
            for seed in SEEDS:
                try:
                    trials.append(fit_trial(waves, mode, width, seed))
                except Exception as error:
                    job = f"{mode}_{width}_{seed}"
                    issues.append(dict(stage="fit", job=job, reason=str(error)))
                    print('CNN unavailable:', job, '[details omitted from notebook output]', flush=True)
training = pd.DataFrame(trials, columns=TRIAL_COLUMNS)
print("completed trials =", len(training), "/", len(MODES) * len(WIDTHS) * len(SEEDS))
display(training)

## 7. 검증 AP로 폭과 대표 seed를 선택한다

작은 팔은 `width ≤ 32`, 중간 팔은 `width ≥ 64`다. 각 정규화·팔의 모든 폭 × seed가 완료되어야 선택한다. 폭은 **세 seed 평균 검증 AP**, 대표 seed는 선택한 폭 안의 검증 AP로 정한다. 동점이면 작은 폭·작은 seed를 고른다.

정규화 효과는 maxabs에서 고른 폭을 고정하여 같은 seed의 maxabs − z로 비교한다. z가 별도로 고른 폭과 비교하면 정규화와 용량의 효과가 섞인다.

In [ ]:
def select_arm(training, mode, arm):
    widths = [width for width in WIDTHS if (width <= 32) == (arm == "small")]
    subset = training.loc[training["mode"].eq(mode) & training["width"].isin(widths)]
    # Example: width 8 has seeds [42, 43] -> the whole small arm is incomplete.
    complete = bool(widths) and all(
        sorted(subset.loc[subset["width"].eq(width), "seed"]) == sorted(SEEDS)
        for width in widths
    )
    if not complete:
        raise ValueError("폭 × seed 탐색이 완료되지 않았습니다")
    means = subset.groupby("width")["val_AP"].mean().reset_index()
    # Example: width 8 mean=0.40, width 32 mean=0.42 -> choose width 32.
    best_width = means.sort_values(["val_AP", "width"], ascending=[False, True]).iloc[0]
    width = int(best_width["width"])
    candidates = subset.loc[subset["width"].eq(width)]
    chosen = candidates.sort_values(["val_AP", "seed"], ascending=[False, True]).iloc[0]
    selected = dict(mode=mode, width=width, seed=int(chosen["seed"]), val_AP=float(chosen["val_AP"]))
    return selected, means.assign(mode=mode, arm=arm)

In [ ]:
# Two widths tie in mean AP; width 8 wins. Its best seed is 43.
example_trials = pd.DataFrame([
    dict(mode="channel_maxabs", width=width, seed=seed, val_AP=ap)
    for width in (8, 32) for seed, ap in ((42, 0.4), (43, 0.5), (44, 0.3))
])
# The example uses the default small-arm widths; the full budget also requires width 16.
if not FULL_CNN:
    example_selection, example_means = select_arm(example_trials, "channel_maxabs", "small")
    display(example_means)
    print("example selected width / seed =", example_selection["width"], example_selection["seed"])

In [ ]:
selection = {}
width_parts = []
for mode in MODES:
    for arm in ("small", "medium"):
        try:
            chosen, means = select_arm(training, mode, arm)
            selection[mode + "_" + arm] = chosen
            width_parts.append(means)
        except ValueError as error:
            issues.append(dict(stage="select", job=mode + "_" + arm, reason=str(error)))
capacity = pd.DataFrame(columns=["width", "val_AP", "mode", "arm"])
if width_parts:
    capacity = pd.concat(width_parts, ignore_index=True)
display(pd.DataFrame(selection).T)

In [ ]:
planned = {}
for chosen in selection.values():
    for seed in SEEDS:
        planned[(chosen["mode"], chosen["width"], seed)] = True
for arm in ("small", "medium"):
    chosen = selection.get("channel_maxabs_" + arm)
    if chosen:
        # Example: maxabs selects width 32 -> evaluate z width 32 at all three same seeds.
        for mode in MODES:
            for seed in SEEDS:
                planned[(mode, chosen["width"], seed)] = True
print("test evaluation jobs =", len(planned))

## 8. 선택한 모델의 시험 예측과 비용을 구한다

`best.pt`를 읽고 validation 추론으로 준비한 뒤 test 추론을 세 번 측정한다. 시간의 중앙값을 시험 구간 수로 나눠 구간당 마이크로초를 구한다. 정규화는 측정에 포함하고 모델 로딩은 제외한다. test 예측은 학습을 다시 하거나 선택을 바꾸는 데 사용하지 않는다.

In [ ]:
def make_prediction(frame, scores):
    # Window rows carry split/signal_index; record rows do not have those columns.
    columns = [column for column in PREDICTION_COLUMNS[:-2] if column in frame]
    prediction = frame[columns].copy()
    # scores[i] belongs to frame.iloc[i], before any ID alignment for comparison.
    prediction["score"] = np.asarray(scores, dtype=float)
    prediction["threshold"] = np.nan  # This experiment does not set alarm thresholds.
    if prediction["row_id"].duplicated().any() or not np.isfinite(prediction["score"]).all():
        raise ValueError("예측 ID가 중복되거나 점수가 유한하지 않습니다")
    return prediction

In [ ]:
def predict_trial(mode, width, seed):
    folder = CNN / f"{mode}_w{width}_s{seed}"
    if not (folder / "complete.json").exists():
        raise ValueError("완료되지 않은 학습은 평가하지 않습니다")
    model = CTGNetMini(width).to(device)
    model.load_state_dict(torch.load(folder / "best.pt", map_location=device, weights_only=True))
    scores(model, normalize(raw[masks["val"]], mode, scale))  # Warm up before timing test inference.
    durations = []
    for _ in range(3):
        started = time.perf_counter()
        probabilities = scores(model, normalize(raw[masks["test"]], mode, scale))
        durations.append(time.perf_counter() - started)
    prediction = make_prediction(test, probabilities)
    # Example: median 0.020 seconds / 1,000 windows -> 20 microseconds per window.
    cost = dict(
        mode=mode, width=width, seed=seed,
        microseconds_per_window=1e6 * np.median(durations) / len(test),
        parameters=sum(value.numel() for value in model.parameters()),
    )
    return prediction, cost

In [ ]:
predictions = {}
cost_rows = []
for mode, width, seed in planned:
    try:
        prediction, cost = predict_trial(mode, width, seed)
        predictions[(mode, width, seed)] = prediction
        cost_rows.append(cost)
    except Exception as error:
        issues.append(dict(stage="predict", job=f"{mode}_{width}_{seed}", reason=str(error)))
costs = pd.DataFrame(cost_rows, columns=COST_COLUMNS)
if predictions:
    sample_key = next(iter(predictions))
    print("example trial (mode, width, seed) =", sample_key)
    print("prediction shape =", predictions[sample_key].shape)

## 9. 같은 시험 산모로 지표와 차이를 계산한다

AP·AUROC·Brier와 95% 산모 bootstrap CI를 계산한다. 04에서 선택한 Cat28 − 선택 CNN을 주 비교로, 같은 seed끼리의 점 차이를 보조 비교로 남긴다. 정규화 비교는 같은 폭·seed의 maxabs − z다.

AP·AUROC 차이는 양수일 때 왼쪽에 유리하고 Brier 차이는 음수일 때 왼쪽에 유리하다. bootstrap은 모델을 재학습하지 않는다.

In [ ]:
def align_rows(left, right):
    # Example: left IDs [A, B], right IDs [B, A] -> reorder right to [A, B].
    # Missing B is an error; comparing only the intersection changes the cohort.
    if left["row_id"].duplicated().any() or right["row_id"].duplicated().any():
        raise ValueError("평가 행 ID가 중복됩니다")
    if set(left["row_id"]) != set(right["row_id"]):
        raise ValueError("비교할 평가 행 집합이 다릅니다")
    # Reordering rows never changes their labels, records, or mother IDs.
    aligned = right.set_index("row_id").loc[left["row_id"]].reset_index()
    for column in ("mother_id", "target", "record_id"):
        if not np.array_equal(left[column].to_numpy(), aligned[column].to_numpy()):
            raise ValueError("평가 행 정보 불일치: " + column)
    return aligned

In [ ]:
def prepare_metrics(prediction):
    labels = prediction["target"].to_numpy(dtype=int)
    scores = prediction["score"].to_numpy(dtype=float)
    if not len(prediction) or set(labels) - {0, 1} or not np.isfinite(scores).all():
        raise ValueError("평가 라벨·점수가 올바르지 않거나 평가 집합이 비었습니다")

    # Example: mother IDs [A, A, B] -> mother_index [0, 0, 1].
    mothers, mother_index = np.unique(prediction["mother_id"], return_inverse=True)
    # Example: scores [0.2, 0.9, 0.5] -> descending row order [1, 2, 0].
    order = np.argsort(-scores, kind="stable")
    # Example: sorted scores [0.9, 0.9, 0.3] -> group ends [1, 2].
    # Tied scores are one threshold, independent of their row order.
    ends = np.r_[np.flatnonzero(np.diff(scores[order])), len(prediction) - 1]
    return dict(
        labels=labels, scores=scores, mothers=mothers,
        mother_index=mother_index, order=order, ends=ends,
    )

In [ ]:
def weighted_metrics(data, counts=None):
    if counts is None:
        # Point estimate: every mother appears once, so every row has weight 1.
        counts = np.ones(len(data["mothers"]))
    # Example: mother counts [2, 0] -> weights [2, 2, 0] for [A, A, B].
    weights = counts[data["mother_index"]]
    labels = data["labels"]
    # Denominators count weighted positive and negative rows.
    positive = np.sum(weights * labels)
    negative = np.sum(weights * (1 - labels))
    # Skip a resample containing only negative or only positive rows.
    if not positive or not negative:
        return None

    order, ends = data["order"], data["ends"]
    # Count rows at or above each distinct score threshold.
    # Example: distinct descending scores with labels [1, 0, 1] -> TP [0, 1, 1, 2].
    true_positive = np.r_[0, np.cumsum((weights * labels)[order])[ends]]
    false_positive = np.r_[0, np.cumsum((weights * (1 - labels))[order])[ends]]
    recall = true_positive / positive
    false_positive_rate = false_positive / negative
    predicted_positive = np.maximum(true_positive[1:] + false_positive[1:], 1e-30)
    # Example: descending labels [1, 0, 1, 0] -> AP = (1 + 2/3)/2 = 5/6.
    ap = np.sum(np.diff(recall) * true_positive[1:] / predicted_positive)
    # Those labels with distinct scores give AUROC=3/4.
    auc = np.sum(np.diff(false_positive_rate) * (recall[1:] + recall[:-1]) / 2)
    # Example: label 1, score 0.9 contributes (0.9 - 1)^2 = 0.01.
    brier = np.sum(weights * (data["scores"] - labels) ** 2) / weights.sum()
    return dict(AP=ap, AUROC=auc, Brier=brier)

In [ ]:
def resample_metrics(left_data, right_data, n_boot, seed):
    samples = {metric: [] for metric in ("AP", "AUROC", "Brier")}
    mother_count = len(left_data["mothers"])
    rng = np.random.default_rng(seed)
    for _ in range(n_boot):
        # Example: mothers [A, A, C] -> counts [2, 0, 1] for [A, B, C].
        # All of A's rows get weight 2 on both sides of the comparison.
        counts = np.bincount(rng.integers(mother_count, size=mother_count), minlength=mother_count)
        left = weighted_metrics(left_data, counts)
        right = weighted_metrics(right_data, counts) if right_data is not None else None
        if left is None or (right_data is not None and right is None):
            continue
        for metric in samples:
            value = left[metric]
            if right is not None:
                value -= right[metric]
            if np.isfinite(value):
                samples[metric].append(value)
    return samples

In [ ]:
def metric_intervals(left, right=None, n_boot=N_BOOTSTRAP, seed=BOOTSTRAP_SEED):
    left_data = prepare_metrics(left)
    right_data = prepare_metrics(align_rows(left, right)) if right is not None else None
    point = weighted_metrics(left_data)
    reference = weighted_metrics(right_data) if right_data is not None else None
    if point is None or (right_data is not None and reference is None):
        raise ValueError("평가에는 양성·음성 행이 모두 필요합니다")
    if reference is not None:
        # Example: Cat28 AP 0.65 - CNN AP 0.60 -> +0.05.
        point = {metric: value - reference[metric] for metric, value in point.items()}
    samples = resample_metrics(left_data, right_data, n_boot, seed)
    rows = []
    for metric, value in point.items():
        low, high = np.nan, np.nan
        if samples[metric]:
            # Central 95% of valid draws; one-class draws do not enter the interval.
            low, high = np.quantile(samples[metric], [0.025, 0.975])
        rows.append(dict(
            metric=metric, point=value, low=low, high=high,
            # n_boot=0 -> point estimate only, NaN endpoints, valid_bootstrap=0.
            valid_bootstrap=len(samples[metric]),
            mothers=len(left_data["mothers"]), segments=len(left),
        ))
    return pd.DataFrame(rows)

In [ ]:
baseline = None
baseline_seeds = {}
if predictions:
    for seed in (None,) + SEEDS:
        path = RESULTS / "pred_cat28.csv" if seed is None else RESULTS / "seeds" / f"pred_cat28_{seed}.csv"
        try:
            saved = pd.read_csv(path, dtype=ID_TYPES)
            saved_test = saved.loc[saved["split"].eq("test")]
            # Verify the whole cohort; never silently compare only intersecting IDs.
            aligned = align_rows(test, saved_test)
            if seed is None:
                baseline = aligned
            else:
                baseline_seeds[seed] = aligned
        except (OSError, ValueError, KeyError) as error:
            issues.append(dict(stage="reference", job=path.name, reason=str(error)))

In [ ]:
absolute_parts, paired_parts, same_seed_parts = [], [], []
for name, chosen in selection.items():
    key = (chosen["mode"], chosen["width"], chosen["seed"])
    if key not in predictions:
        continue
    try:
        absolute_parts.append(metric_intervals(predictions[key]).assign(model=name))
        if baseline is not None:
            difference = metric_intervals(baseline, predictions[key])
            paired_parts.append(difference.assign(comparison="Cat28 - " + name))
    except Exception as error:
        issues.append(dict(stage="evaluate", job=name, reason=str(error)))
    for seed in SEEDS:
        key = (chosen["mode"], chosen["width"], seed)
        if seed not in baseline_seeds or key not in predictions:
            continue
        try:
            # Same-seed comparison has a point estimate only: n_boot=0.
            difference = metric_intervals(baseline_seeds[seed], predictions[key], n_boot=0)
            same_seed_parts.append(difference.assign(comparison="Cat28 - " + name, seed=seed))
        except Exception as error:
            issues.append(dict(stage="same_seed", job=f"{name}_{seed}", reason=str(error)))

In [ ]:
normalization_parts = []
for arm in ("small", "medium"):
    chosen = selection.get("channel_maxabs_" + arm)
    if chosen is None:
        continue
    for seed in SEEDS:
        left = ("channel_maxabs", chosen["width"], seed)
        right = ("per_segment_z", chosen["width"], seed)
        if left not in predictions or right not in predictions:
            continue
        try:
            # Both scores use exactly the same width, seed, windows and bootstrap draws.
            difference = metric_intervals(predictions[left], predictions[right])
            normalization_parts.append(difference.assign(
                comparison=arm + ": maxabs - z", width=chosen["width"], seed=seed,
            ))
        except Exception as error:
            issues.append(dict(stage="normalization", job=f"{arm}_{seed}", reason=str(error)))

In [ ]:
# Define empty schemas too, so a failed rerun replaces old summary results.
result_parts = {
    "metrics": (absolute_parts, ["model"]),
    "paired": (paired_parts, ["comparison"]),
    "same_seed": (same_seed_parts, ["comparison", "seed"]),
    "normalization": (normalization_parts, ["comparison", "width", "seed"]),
}
results = {}
for name, (parts, extra_columns) in result_parts.items():
    results[name] = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=METRIC_COLUMNS + extra_columns)
    print(name)
    display(results[name])

In [ ]:
def plot_difference(table, metric, title):
    selected = table.loc[table["metric"].eq(metric)]
    if selected.empty:
        print("그릴 비교 결과가 없습니다.")
        return
    positions = np.arange(len(selected))
    figure, axis = plt.subplots(figsize=(8, max(3, len(selected) * 0.4)))
    # Each dot is left-minus-right; horizontal lines are mother-bootstrap CIs.
    axis.hlines(positions, selected["low"], selected["high"], color="C0")
    axis.scatter(selected["point"], positions)
    axis.axvline(0, color="gray", linestyle="--")
    axis.set_yticks(positions, selected["comparison"])
    axis.set_xlabel(metric + " difference (left minus right)")
    axis.set_title(title)
    figure.tight_layout()
    plt.show()

In [ ]:
plot_difference(results["paired"], "AP", "Cat28 minus CNN: same test mothers")

In [ ]:
plot_difference(results["paired"], "AUROC", "Cat28 minus CNN: same test mothers")

## 10. 용량·학습 곡선·비용을 읽는다

폭별 평균 validation AP와 선택 모델의 train loss·validation loss·AP를 함께 본다. `hit_cap=True`이면서 `patience_met=False`이면 예산 상한에 도달했으나 정체 조건에는 도달하지 못한 것이다. `patience_met=True`도 전역 최적점의 증거는 아니다.

seed별 차이는 학습의 변동, 산모 CI는 고정 모델에 대한 평가 표본의 변동이다. 둘을 서로 대신 해석하지 않는다.

In [ ]:
if not capacity.empty:
    figure, axis = plt.subplots(figsize=(7, 4))
    for mode, group in capacity.groupby("mode"):
        axis.plot(group["width"], group["val_AP"], marker="o", label=mode)
    axis.set(xlabel="Width", ylabel="Mean validation AP across 3 seeds")
    axis.legend()
    figure.tight_layout()
    plt.show()

In [ ]:
def plot_history(name, chosen):
    folder = CNN / f"{chosen['mode']}_w{chosen['width']}_s{chosen['seed']}"
    history = pd.read_csv(folder / "history.csv")
    complete = json.loads((folder / "complete.json").read_text(encoding="utf-8"))
    figure, axes = plt.subplots(1, 2, figsize=(11, 3))
    axes[0].plot(history["epoch"], history["train_loss"], label="train")
    axes[0].plot(history["epoch"], history["val_loss"], label="validation")
    axes[0].set(title=name, ylabel="Weighted BCE", xlabel="Epoch")
    axes[0].legend()
    axes[1].plot(history["epoch"], history["val_AP"])
    # Example: best_epoch=20 draws a line at 20, even if training ran to epoch 60.
    axes[1].axvline(complete["best_epoch"], color="gray", linestyle="--")
    axes[1].set(ylabel="Validation AP", xlabel="Epoch")
    figure.tight_layout()
    plt.show()

In [ ]:
for name, chosen in selection.items():
    try:
        plot_history(name, chosen)
    except (OSError, ValueError, KeyError) as error:
        issues.append(dict(stage="plot", job=name, reason=str(error)))

In [ ]:
display(costs)
# Show feature extraction and Cat28 model timing separately, when available.
# These timings do not include all disk I/O or deployment overhead.
for filename in ("extraction_timing.json", "04_training.csv"):
    path = RESULTS / filename
    if path.exists():
        try:
            print(filename)
            display(json.loads(path.read_text()) if path.suffix == ".json" else pd.read_csv(path))
        except (OSError, ValueError) as error:
            issues.append(dict(stage="cost_reference", job=filename, reason=str(error)))

## 11. 저장 파일과 완료 상태

학습 중에는 `cnn/{정규화}_w{폭}_s{seed}/`에 `best.pt`, `last.pt`, `history.csv`, `complete.json`을 저장한다. 시험 평가 대상 폴더에는 `test_predictions.csv`도 저장한다.

| 파일 | 내용 |
|---|---|
| `cnn/run.json` | 체크포인트 재사용을 위한 입력·설정 확인 |
| `cnn_normalization.json` | 훈련 채널 분모와 장치 |
| `09_training.csv` | 조합별 최적 AP·epoch·중단 상태·학습 시간 |
| `09_selection.json`, `09_capacity.csv` | 선택 폭·대표 seed와 폭별 평균 검증 AP |
| `09_metrics.csv`, `09_paired.csv` | 선택 CNN 성능과 Cat28 − CNN 차이·CI |
| `09_same_seed.csv` | 같은 seed의 Cat28 − CNN 점 차이 |
| `09_normalization.csv` | 같은 폭·seed의 maxabs − z 차이·CI |
| `09_costs.csv` | 시험 추론 시간과 파라미터 수 |
| `09_status.json`, `09_issues.csv` | 완료 상태·예산 소진·미산출 사유 |

요약 파일은 결과가 없더라도 빈 열 구조로 갱신한다. 입력 확인에 실패한 경우 기존 모델 폴더는 유지하며, 이번 실행의 미산출 상태를 저장한다. `fit_seconds`와 추론 시간은 실행 환경에 따라 달라진다.

In [ ]:
RESULTS.mkdir(parents=True, exist_ok=True)
training.to_csv(RESULTS / "09_training.csv", index=False)
capacity.to_csv(RESULTS / "09_capacity.csv", index=False)
costs.to_csv(RESULTS / "09_costs.csv", index=False)
for name, table in results.items():
    table.to_csv(RESULTS / f"09_{name}.csv", index=False)
(RESULTS / "09_selection.json").write_text(json.dumps(selection, indent=2), encoding="utf-8")
normalization_state = dict(scale=scale.tolist(), device=str(device)) if ready else dict(scale=None, device=None, error=setup_error)
(RESULTS / "cnn_normalization.json").write_text(json.dumps(normalization_state, ensure_ascii=False, indent=2), encoding="utf-8")

for key in planned:
    mode, width, seed = key
    folder = CNN / f"{mode}_w{width}_s{seed}"
    # Replace stale test rows after a failed inference job; keep training checkpoints.
    prediction = predictions.get(key, pd.DataFrame(columns=PREDICTION_COLUMNS))
    prediction.to_csv(folder / "test_predictions.csv", index=False)

In [ ]:
issue_table = pd.DataFrame(issues, columns=["stage", "job", "reason"])
issue_table.to_csv(RESULTS / "09_issues.csv", index=False)
unresolved = [trial for trial in trials if trial["hit_cap"] and not trial["patience_met"]]
status = dict(
    completed_trials=len(trials), expected_trials=len(MODES) * len(WIDTHS) * len(SEEDS),
    selected=selection, cap_without_patience=unresolved,
    setup_error=setup_error, predicted_trials=len(predictions), issue_count=len(issues),
)
(RESULTS / "09_status.json").write_text(json.dumps(status, ensure_ascii=False, indent=2), encoding="utf-8")
print('saved to =', RESULTS)
print("completed / expected trials =", status["completed_trials"], "/", status["expected_trials"])
print("hit cap without patience =", len(unresolved))
if not issue_table.empty:
    print("issue count =", len(issue_table))

## 해석의 범위

Cat28とCNNの差は、試験指標とCIだけでなく学習状態も合わせて読む。CIが0を含むことだけで同等性や非劣性を宣言しない。

선택한 폭에서의 정규화 차이를 모든 폭의 효과로 일반화하지 않는다. 이 실험은 저장소 구조를 재구현한 작은 CNN 대조이며, 기존 연구와 데이터·입력 길이가 같은 재현 실험으로 보지 않는다. 과거 연구에서 이미 본 산모인지 확인되지 않은 경우 독립적인 외부 검증으로 해석하지 않는다.

### 반출용 제외·오류 집계

개별 ID·경로·오류 전문을 제외한 분류별 건수다.

In [ ]:
show_issue_counts(issue_table["reason"])